# Conduction velocity in plants — two-channel deep dive

How does a propagating wound / variation potential change as it travels from the
**near** electrode (channel 1) to the **far** electrode (channel 2), and what
conduction velocity does that imply?

**Protocol.** A stimulus is applied at one site; two electrodes sit inline
downstream at different distances. The two event markers bracket the
*stimulation window* (start/stop) — the response is analysed afterwards. Where
known, the inter-electrode distance is in the filename (`-28.8mm`).


In [ ]:
import os, sys, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath(".."))
from cvplants.io import load_recording, iter_dataset
from cvplants.analysis import analyze_recording
from cvplants.batch import build_results, species_summary
from cvplants import viz

DATA = os.path.abspath("../data")
np.random.seed(0)
sorted(os.listdir(DATA))

## 1. One recording: the two channels and the stimulation window

A *Cannabis* recording with a known 21 mm electrode spacing. The near channel (blue) peaks first; the far channel (red) follows — that lag is the propagation delay.

In [ ]:
rec = load_recording(os.path.join(DATA, "Marijuana",
        "BYB_Recording_2025-08-23_23.19.28-21mm.wav"), species="Marijuana")
fig, ax = plt.subplots(figsize=(11,4))
res = viz.plot_recording(rec, ax=ax)
plt.show()
{k: res[k] for k in ["distance_mm","xcorr_delay_s","peak_delay_s",
                     "cv_xcorr_mm_s","attenuation_far_near","broadening_far_near","flags"]}

## 2. Why the markers are not the arrival times

The markers are ~7 s apart, but the waveform lag between channels is only ~1–3 s.
The 7 s is the *stimulation duration*; conduction velocity comes from the
response waveforms, not the markers. The two independent delay estimates
(cross-correlation vs peak-to-peak) agree — that is the check that the automatic
delay is trustworthy.

In [ ]:
df = build_results(DATA)
mj = df[(df.species=="Marijuana") & df.valid]
ax = mj.plot.scatter("peak_delay_s","xcorr_delay_s", s=40)
lim=[0, mj[["peak_delay_s","xcorr_delay_s"]].max().max()*1.1]
ax.plot(lim,lim,"r--"); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_title("Cannabis: peak-delay vs cross-correlation delay agree"); plt.show()

## 3. How the signal changes from near to far

Across all valid recordings: the potential **attenuates** (far/near amplitude < 1), **broadens** (far/near FWHM > 1), while the **shape is preserved** (high waveform correlation).

In [ ]:
viz.plot_transformation(df, "_transformation.png")
from matplotlib import image as mpimg
plt.figure(figsize=(14,4.3)); plt.imshow(mpimg.imread("_transformation.png")); plt.axis("off"); plt.show()

## 4. Conduction velocity by species

Absolute CV where the electrode distance is known (Cannabis, and a few Venus flytrap).

In [ ]:
summary = species_summary(df)
summary[["species","latin","n_valid","cv_xcorr_median","delay_median_s",
         "attenuation_median","broadening_median","waveform_corr_median"]]

In [ ]:
viz.plot_cv_by_species(df, "_cv.png")
plt.figure(figsize=(11,6)); plt.imshow(mpimg.imread("_cv.png")); plt.axis("off"); plt.show()

## 5. Distance vs delay (Cannabis)

Slope through the origin is an aggregate conduction velocity. The scatter shows that within one plant the delay is not a simple linear function of electrode distance — biological CV variability dominates.

In [ ]:
viz.plot_distance_delay(df, "_dd.png", species="Marijuana")
plt.figure(figsize=(6.5,5)); plt.imshow(mpimg.imread("_dd.png")); plt.axis("off"); plt.show()

## 6. Per-species example traces

In [ ]:
recs = list(iter_dataset(DATA))
viz.plot_species_grid(recs, "_grid.png")
plt.figure(figsize=(11,30)); plt.imshow(mpimg.imread("_grid.png")); plt.axis("off"); plt.show()

## Notes

- Amplitudes are raw ADC units (a.u.); ratios and timing are unaffected.
- Add `-<mm>mm` to filenames (or extend `parse_distance_mm`) to unlock absolute
  CV for the remaining species.
- Venus flytrap fires fast action potentials, not slow variation potentials —
  treat its velocities as a separate signal class.